# Kimi K3 recreation — train on CodeParrot (Colab, 1× T4)

This notebook trains the **Kimi K3 recreation** (Block Attention Residuals + Gated MLA + LatentMoE,
with Gated DeltaNet-2 standing in for Kimi Delta Attention) on the **CodeParrot** Python-code corpus.

**Google Colab** — Runtime ▸ Change runtime type ▸ **T4 GPU**.

Pipeline (all code lives in the repo, this notebook only drives it):
1. **Grain** — deterministic packed-sequence data loading (`codeparrot_data.py`)
2. **Optax** — AdamW, warmup-cosine, grad clip + MoE router-bias balancing (`train.py`)
3. **Orbax** — checkpointing with automatic resume (`train.py` / `evaluate.py`)

> **T4 note:** T4s (Turing) have no native bfloat16, so we keep the default `compute_dtype="float32"`.


## 1. Get the code
Push this project to GitHub first, then set `REPO_URL`.

In [ ]:
import os, pathlib

REPO_URL = "https://github.com/wisnunugroho21/nugie-kimi-k3.git"   # <-- EDIT ME

if not pathlib.Path("nugie-kimi-k3").exists():
    !git clone {REPO_URL} nugie-kimi-k3
%cd nugie-kimi-k3

## 2. Install dependencies
JAX with CUDA wheels, plus Grain / Optax / Orbax and the HF data stack.

In [ ]:
%pip install -q -U "jax[cuda12]" flax grain optax orbax-checkpoint datasets transformers

import jax
print("devices:", jax.devices())

## 3. (Optional) Hugging Face token
Add an `HF_TOKEN` secret in the Colab sidebar (🔑) to speed up streaming.

In [ ]:
# Optional: HF token for faster (authenticated) Hub streaming.
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN set")
except Exception:
    print("no HF_TOKEN secret — continuing unauthenticated (fine, just slower)")

## 4. Prepare the CodeParrot data
Streams `codeparrot/codeparrot-clean` (no 50 GB download), tokenizes with the official
`codeparrot/codeparrot` BPE (vocab 32768), packs into `train.bin` / `val.bin` memmaps.
~20M tokens takes a few minutes; raise the budgets for longer runs.

In [ ]:
!python codeparrot_data.py --out data --train-tokens 20000000 --val-tokens 500000

## 5. Train
Re-running this cell (or a fresh session with the same `checkpoints/` dir) **auto-resumes** from the latest Orbax checkpoint — pass a larger `--steps` to extend the run.

In [ ]:
!python train.py --data-dir data --ckpt-dir checkpoints \
    --steps 3000 --batch-size 32 --seq-len 512 \
    --lr 3e-4 --warmup-steps 200 \
    --log-every 25 --eval-every 500 --eval-batches 20 --ckpt-every 500

## 6. Evaluate + sample
Validation cross-entropy / perplexity over the held-out CodeParrot split, then a greedy
code completion through the streaming decode path (GDN-2 recurrent state + MLA latent cache).

In [ ]:
!python evaluate.py --ckpt-dir checkpoints --prompt "def fibonacci(n):" --sample-tokens 64

## 7. (Optional) Persist checkpoints to Drive
Colab VMs are ephemeral — copy `checkpoints/` to Drive to resume later.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r checkpoints /content/drive/MyDrive/kimi_k3_checkpoints
# To resume in a new session, copy it back BEFORE the train cell:
# !cp -r /content/drive/MyDrive/kimi_k3_checkpoints checkpoints

## Scaling notes
* `--batch-size` / `--seq-len` — the T4 has plenty of headroom at the default 49M-param model;
  try `--batch-size 64` (Colab) / `128` (Kaggle) if you raise `--steps`.
  `seq_len` must stay a multiple of 64 (the GDN-2 chunk size).
* Model size — edit the `KimiK3Config` defaults in `kimi_k3_gdn2.py` (`d_model`, `n_layers`,
  `moe_n_routed`, ...). Keep `moe_d_latent ≈ d_model/4` (the LatentMoE recipe).
* Longer data — raise `--train-tokens` in step 4; the packed `.bin` files are reused across runs.